# Phase 3 — Baselines

[`walkthrough/phase3.md`](../walkthrough/phase3.md). Training split only, PSU-grouped 5-fold CV.
The single income threshold is re-learned inside every training fold, so no fold is scored
with a cut-off that saw it.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

from savings_goal import config as C
from savings_goal import viz
from savings_goal.evaluation.cv import cross_validate_grouped, grouped_cv
from savings_goal.evaluation.metrics import fold_metrics
from savings_goal.features.engineer import spec_from_frame
from savings_goal.io import load_features, write_result
from savings_goal.models.pipeline import logistic, preprocessor

viz.use()
feats = load_features()
spec = spec_from_frame(feats)
train = feats[~feats["Is_Test"]].reset_index(drop=True)
y, groups = train["Goal_Met"], train[C.GROUP_COL]

METRICS = {
    **fold_metrics(with_proba=False),
    "precision": lambda yt, s, yp: precision_score(yt, yp, zero_division=0),
    "recall": lambda yt, s, yp: recall_score(yt, yp),
}
print(f"training split: {len(train):,} households; Goal_Met rate {y.mean():.4f}")


def evaluate(model, cols, label, on_fold=None):
    folds = cross_validate_grouped(model, train[cols], y, groups, METRICS, grouped_cv(), on_fold)
    return {"model": label, **folds.mean().to_dict(), "F1 sd": folds["F1 (macro)"].std()}

training split: 33,219 households; Goal_Met rate 0.3192


## Q1 — Majority-class and single-rule baselines

In [2]:
ALL = spec.features
rows = [
    evaluate(DummyClassifier(strategy="most_frequent"), ALL, "Majority class (always 'not met')"),
    evaluate(DummyClassifier(strategy="stratified", random_state=C.SEED), ALL, "Stratified random"),
]

thresholds = []


def record_threshold(k, model, train_idx):
    thresholds.append(float(np.exp(model[-1].tree_.threshold[0])))


stump = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=1, random_state=C.SEED))
rows.append(evaluate(stump, ["Log_Income"], "Single income threshold (depth-1 tree)", record_threshold))
rows.append(evaluate(make_pipeline(SimpleImputer(strategy="median"),
                                   DecisionTreeClassifier(max_depth=3, random_state=C.SEED)),
                     ["Log_Income"], "Income only (depth-3 tree)"))
baseline = pd.DataFrame(rows).set_index("model")
print(baseline.round(4).to_string())
thr = np.array(thresholds)
print(f"\nincome threshold learned per training fold: Rs {thr.mean():,.0f}/yr "
      f"(range Rs {thr.min():,.0f} - {thr.max():,.0f})")

                                        accuracy  F1 (macro)  ROC-AUC  PR-AUC (at-risk)  precision  recall   F1 sd
model                                                                                                             
Majority class (always 'not met')         0.6808      0.4050   0.5000            0.6808     0.0000  0.0000  0.0001
Stratified random                         0.5698      0.5022   0.5023            0.6817     0.3224  0.3155  0.0037
Single income threshold (depth-1 tree)    0.7781      0.7417   0.7392            0.8077     0.6600  0.6316  0.0040
Income only (depth-3 tree)                0.7813      0.7268   0.8273            0.8874     0.7169  0.5259  0.0059

income threshold learned per training fold: Rs 122,249/yr (range Rs 110,377 - 126,545)


## Q2 — Logistic regression on income plus a few features

In [3]:
MINIMAL = {
    "income only": ["Log_Income"],
    "income + groceries share": ["Log_Income", "Groceries_Share"],
    "income + household size": ["Log_Income", "Household_Size"],
    "income + size + groceries share": ["Log_Income", "Household_Size", "Groceries_Share"],
    "deployable (income, demographics, debt)": spec.deployable_numeric,
    f"all {len(spec.features)} features": spec.numeric,
}
rows = []
for label, num in MINIMAL.items():
    cats = C.CATEGORICALS if len(num) > 3 else []
    model = make_pipeline(preprocessor(num, cats, linear=True), logistic())
    rows.append(evaluate(model, num + cats, label))
progression = pd.DataFrame(rows).set_index("model")
print(progression.round(4).to_string())
auc = progression["ROC-AUC"]
print(f"\nGroceries_Share adds {auc['income + groceries share'] - auc['income only']:+.4f} ROC-AUC to income "
      f"alone and {auc['income + size + groceries share'] - auc['income + household size']:+.4f} once "
      "household size is in: size x food share x income is the reconstruction route found in Phase 1.")

                                         accuracy  F1 (macro)  ROC-AUC  PR-AUC (at-risk)  precision  recall   F1 sd
model                                                                                                              
income only                                0.7519      0.7319   0.8345            0.9104     0.5873  0.7499  0.0063
income + groceries share                   0.7877      0.7701   0.8752            0.9314     0.6325  0.8001  0.0055
income + household size                    0.7699      0.7500   0.8496            0.9223     0.6119  0.7636  0.0077
income + size + groceries share            0.8059      0.7890   0.8938            0.9436     0.6574  0.8187  0.0077
deployable (income, demographics, debt)    0.7911      0.7731   0.8763            0.9378     0.6378  0.7991  0.0068
all 29 features                            0.8350      0.8193   0.9200            0.9597     0.6997  0.8465  0.0089

Groceries_Share adds +0.0407 ROC-AUC to income alone and +0.0443 once h

In [4]:
full = pd.concat([baseline, progression])
full.to_csv(C.RESULTS / "baseline.csv")
write_result("baseline", {
    "income_threshold_rs_per_fold": thr, "income_threshold_rs_mean": float(thr.mean()),
    "table": full.reset_index(), "split": "training split, PSU-grouped 5-fold CV",
})

short = {"Majority class (always 'not met')": "Majority class", "Stratified random": "Stratified random",
         "Single income threshold (depth-1 tree)": "Income threshold (depth 1)",
         "Income only (depth-3 tree)": "Income tree (depth 3)", "income only": "LR: income",
         "income + groceries share": "LR: income + food share", "income + household size": "LR: income + size",
         "income + size + groceries share": "LR: income + size + food share",
         "deployable (income, demographics, debt)": "LR: income + demographics + debt"}
plot = full.rename(index=lambda m: short.get(m, f"LR: {m}"))
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)
fig.subplots_adjust(wspace=0.08)
maj = plot.loc["Majority class"]
for ax, metric in zip(axes, ["accuracy", "F1 (macro)"]):
    colours = [viz.CONTEXT if m == "Majority class" else viz.BLUE for m in plot.index]
    viz.barh(ax, list(plot.index), list(plot[metric]), colors=colours, xlim=(0, 1.08))
    ax.set_xlabel(metric if metric == "accuracy" else "macro-F1")
axes[1].tick_params(axis="y", labelleft=False)
axes[0].set_title(f"Accuracy: a constant prediction already scores {maj['accuracy']:.2f}")
axes[1].set_title(f"Macro-F1: the same constant scores {maj['F1 (macro)']:.2f}")
plt.savefig(C.RESULTS / "baseline.png")
plt.show()